# Whole Tumor Based Patching Pipeline
1. In folder Path the Link of Parent folder COntaining All modalities should be given.
2. The extensions and name of extension of each file in subfolder must be checked and compared with code for error free execution
3. This code will generate 128x128x128 Patches for all modalities in the same folder and subfolder given as a link in parent directory as it replaces the files (Volumes with Patches).

In [1]:
import numpy as np
import skimage.filters 
import skimage
from skimage import measure
import SimpleITK as sitk
import numpy as np
import os
import numpy as np
import nibabel as nib
import glob
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from tifffile import imsave
import zipfile
import tempfile
from skimage import measure
from concurrent.futures import ProcessPoolExecutor
import time

In [4]:
#It will do the tumor centered cropping and override the existing files. 

start_time = time.time()

folder_path = r"F:\BraTs 2024 Data Unpatched Complete Data\Testing"


def extract_patch(data, centroid):
    data_shape = np.array(data.shape)

    start_idx = np.array(centroid) - np.array([64, 64, 64])
    end_idx   = np.array(centroid) + np.array([64, 64, 64])

    for i in range(3):
        if start_idx[i] < 0:
            end_idx[i] -= start_idx[i]
            start_idx[i] = 0
        if end_idx[i] > data_shape[i]:
            start_idx[i] -= end_idx[i] - data_shape[i]
            end_idx[i] = data_shape[i]

    patch_size = end_idx - start_idx
    patch = np.zeros(patch_size)

    patch[
        0:patch_size[0],
        0:patch_size[1],
        0:patch_size[2]
    ] = data[
        start_idx[0]:end_idx[0],
        start_idx[1]:end_idx[1],
        start_idx[2]:end_idx[2]
    ]

    return patch


# MAIN LOOP
for subdir, dirs, files in os.walk(folder_path):

    centroid1 = None  # reset per case

    # STEP 1: Compute centroid from T2W and save T2W patch to output folder
    for file in files:
        if '-t2w.nii' in file:
            file_path = os.path.join(subdir, file)

            img = sitk.ReadImage(file_path)
            t2w_data = sitk.GetArrayFromImage(img)

            blurred_volume = skimage.filters.gaussian(t2w_data, sigma=1.0)

            new_vol = blurred_volume[56:184, 56:184, 56:184]
            thresh = skimage.filters.threshold_yen(new_vol)

            binary_volume = blurred_volume > thresh

            labels = measure.label(binary_volume, connectivity=2)
            regions = measure.regionprops(labels)
            if len(regions) == 0:
                continue

            biggest_region = max(regions, key=lambda x: x.area)
            biggest_component_mask = (labels == biggest_region.label).astype('uint8')

            image = biggest_component_mask * t2w_data

            if np.count_nonzero(image) == 0:
                continue

            centroid = np.mean(np.nonzero(image), axis=1)
            centroid1 = np.rint(centroid).astype(int)

            # Save T2W patch — overwrite original
            patch = extract_patch(t2w_data, centroid1)
            sitk.WriteImage(
                sitk.GetImageFromArray(patch),
                file_path                         #  overwrite original
            )

    # Skip if no centroid found for this case
    if centroid1 is None:
        continue

    # STEP 2: ALL OTHER MODALITIES — overwrite originals
    for file in files:
        file_path = os.path.join(subdir, file)

        if any(x in file for x in ['-t2f.nii', '-t1c.nii', '-t1n.nii']):
            img = sitk.ReadImage(file_path)
            data = sitk.GetArrayFromImage(img)

            patch = extract_patch(data, centroid1)

            sitk.WriteImage(
                sitk.GetImageFromArray(patch),
                file_path                         #  overwrite original
            )

        elif '-seg.nii' in file:
            img = sitk.ReadImage(file_path)
            data = sitk.GetArrayFromImage(img)

            patch = extract_patch(data, centroid1)

            sitk.WriteImage(
                sitk.GetImageFromArray(patch),
                file_path                         #  overwrite original
            )

end_time = time.time()
print("Total processing time:", end_time - start_time, "seconds")

Total processing time: 2455.4299829006195 seconds


In [ ]:
#It will do the tumor centered cropping and save the cropped volumes to the output folder. 



start_time = time.time()
folder_path   = r"F:\BraTs 2024 Data Unpatched\Training-Data"
output_folder = r"F:\BraTs 2024 Data Tumor Centered Patched"   #your empty output folder


def extract_patch(data, centroid):
    data_shape = np.array(data.shape)

    start_idx = np.array(centroid) - np.array([64, 64, 64])
    end_idx   = np.array(centroid) + np.array([64, 64, 64])

    for i in range(3):
        if start_idx[i] < 0:
            end_idx[i] -= start_idx[i]
            start_idx[i] = 0
        if end_idx[i] > data_shape[i]:
            start_idx[i] -= end_idx[i] - data_shape[i]
            end_idx[i] = data_shape[i]

    patch_size = end_idx - start_idx
    patch = np.zeros(patch_size)

    patch[
        0:patch_size[0],
        0:patch_size[1],
        0:patch_size[2]
    ] = data[
        start_idx[0]:end_idx[0],
        start_idx[1]:end_idx[1],
        start_idx[2]:end_idx[2]
    ]

    return patch


# MAIN LOOP
for subdir, dirs, files in os.walk(folder_path):

    centroid1 = None  # reset per case

    # Build the mirrored output subfolder path
    relative_path = os.path.relpath(subdir, folder_path)
    output_subdir = os.path.join(output_folder, relative_path)
    os.makedirs(output_subdir, exist_ok=True)

    # STEP 1: Compute centroid from T2W and save T2W patch to output folder
    for file in files:
        if '-t2w.nii' in file:
            file_path        = os.path.join(subdir, file)           # original (read only)
            output_file_path = os.path.join(output_subdir, file)    # output (write here)

            img = sitk.ReadImage(file_path)
            t2w_data = sitk.GetArrayFromImage(img)

            blurred_volume = skimage.filters.gaussian(t2w_data, sigma=1.0)

            new_vol = blurred_volume[56:184, 56:184, 56:184]
            thresh = skimage.filters.threshold_yen(new_vol)

            binary_volume = blurred_volume > thresh

            labels = measure.label(binary_volume, connectivity=2)
            regions = measure.regionprops(labels)
            if len(regions) == 0:
                continue

            biggest_region = max(regions, key=lambda x: x.area)
            biggest_component_mask = (labels == biggest_region.label).astype('uint8')

            image = biggest_component_mask * t2w_data

            if np.count_nonzero(image) == 0:
                continue

            centroid = np.mean(np.nonzero(image), axis=1)
            centroid1 = np.rint(centroid).astype(int)

            # Save T2W patch to output folder
            patch = extract_patch(t2w_data, centroid1)
            sitk.WriteImage(
                sitk.GetImageFromArray(patch),
                output_file_path                  # ← output, not original
            )

    # Skip if no centroid found for this case
    if centroid1 is None:
        continue

    # STEP 2: ALL OTHER MODALITIES — save to output folder
    for file in files:
        file_path        = os.path.join(subdir, file)           # original (read only)
        output_file_path = os.path.join(output_subdir, file)    # output (write here)

        if any(x in file for x in ['-t2f.nii', '-t1c.nii', '-t1n.nii']):
            img = sitk.ReadImage(file_path)
            data = sitk.GetArrayFromImage(img)

            patch = extract_patch(data, centroid1)

            sitk.WriteImage(
                sitk.GetImageFromArray(patch),
                output_file_path                  # output, not original
            )

        elif '-seg.nii' in file:
            img = sitk.ReadImage(file_path)
            data = sitk.GetArrayFromImage(img)

            patch = extract_patch(data, centroid1)

            sitk.WriteImage(
                sitk.GetImageFromArray(patch),
                output_file_path                  #  output, not original
            )

end_time = time.time()
print("Total processing time:", end_time - start_time, "seconds")

Total processing time: 4206.4005489349365 seconds
